In [ ]:
import pandas as pd
import os
import cohere
import openai
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, PointStruct, PayloadSchemaType, SparseVectorParams, Document, Prefetch, FusionQuery

/Users/vivekkaushik/Desktop/amazon-chat-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Retrieval

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [12]:
from dotenv import load_dotenv

load_dotenv()
import os
os.environ['CO_API_KEY']=os.getenv("CO_API_KEY")

In [3]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        model=model,
        input=[text]
    )
    return response.data[0].embedding

In [5]:
def retrieve_data(query, qdrant_client, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using='text-embedding-3-small',
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query, 
                    model="qdrant/bm25"),
                using='bm25',
                limit=20

            )
        ],
        query= FusionQuery(fusion='rrf'),
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for point in search_result.points:
        retrieved_context_ids.append(point.payload["parent_asin"])
        retrieved_context.append(point.payload["description"])
        similarity_scores.append(point.score)
        retrieved_context_ratings.append(point.payload["average_rating"])
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [8]:
query = "Can i have a laptop?"

In [9]:
results = retrieve_data(query, qdrant_client, top_k=20)

In [10]:
results

{'retrieved_context_ids': ['B09MQXN33J',
  'B0BXWHB1B9',
  'B09WVM6N7G',
  'B09X33VMY7',
  'B0BMGTY5VX',
  'B0BGR4TDW4',
  'B0BLV5HVSH',
  'B09V2BQ7NW',
  'B0B7QZQQFX',
  'B0BM6W3VPP',
  'B0BZ8BM744',
  'B0BN2FG7MJ',
  'B0BJXS8PJL',
  'B0B8V1BL1N',
  'B0BBD333SD',
  'B0BY33JRYT',
  'B0B772R1YS',
  'B0BTJ8V312',
  'B0CDGT43N4',
  'B09QY84D1S'],
 'retrieved_context': ['LORYERGO Laptop Stand, Adjustable Laptop Stand for Desk, Ergonomic Computer Stand Multi-Angle Adjustable Height, Portable Foldable Laptop Riser Holder Stand Compatible for 10 to 15.6" Laptops 10 Adjustable Angles - The laptop stand has an comfort design that elevates your laptop at an adjustable height for the right angle, you can improve your positioning to work more comfortably. 360°Rotation Base - The laptop stand for desk is equipped with a 360°rotation base, you can turn your laptop to share your screen and quickly change viewing angles. Stable and Secure - The laptop stand can support even heavier laptops up to 15.6 

### Reranking

In [13]:
cohere_client = cohere.ClientV2()

In [14]:
to_rerank = results["retrieved_context"]

In [16]:
to_rerank

['LORYERGO Laptop Stand, Adjustable Laptop Stand for Desk, Ergonomic Computer Stand Multi-Angle Adjustable Height, Portable Foldable Laptop Riser Holder Stand Compatible for 10 to 15.6" Laptops 10 Adjustable Angles - The laptop stand has an comfort design that elevates your laptop at an adjustable height for the right angle, you can improve your positioning to work more comfortably. 360°Rotation Base - The laptop stand for desk is equipped with a 360°rotation base, you can turn your laptop to share your screen and quickly change viewing angles. Stable and Secure - The laptop stand can support even heavier laptops up to 15.6 inches within 11LBS of weight capacity. The adjustable laptop stand with anti-slip paddings and anti-skid baffles will hold your device firmly. Lightweight and Portable - The laptop riser is thin for storage or transport. Ideal for use both indoors and outdoors, creating a workspace anywhere you need one with this laptop stand. Cools Your Laptop - The computer stand

In [18]:
response = cohere_client.rerank(
    model="rerank-v3.5",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [19]:
response

V2RerankResponse(id='46bd084a-e6f8-48e7-8992-1ed0fcf0c253', results=[V2RerankResponseResultsItem(index=10, relevance_score=0.26445615), V2RerankResponseResultsItem(index=0, relevance_score=0.22334355), V2RerankResponseResultsItem(index=7, relevance_score=0.22321655), V2RerankResponseResultsItem(index=8, relevance_score=0.22131741), V2RerankResponseResultsItem(index=19, relevance_score=0.21581353), V2RerankResponseResultsItem(index=2, relevance_score=0.21484825), V2RerankResponseResultsItem(index=9, relevance_score=0.18705876), V2RerankResponseResultsItem(index=12, relevance_score=0.18409216), V2RerankResponseResultsItem(index=3, relevance_score=0.18151005), V2RerankResponseResultsItem(index=6, relevance_score=0.1795165), V2RerankResponseResultsItem(index=4, relevance_score=0.16972247), V2RerankResponseResultsItem(index=5, relevance_score=0.15886505), V2RerankResponseResultsItem(index=14, relevance_score=0.1489489), V2RerankResponseResultsItem(index=17, relevance_score=0.14483722), V2Re

In [20]:
reranked_results = [to_rerank[result.index] for result in response.results ]

In [21]:
reranked_results 

['HUANUO Laptop Stand, Ergonomic Laptop Stand for Desk, Notebook Computer Stand Holder Compatible with 10-15.6 Inch Laptops, Black, HNLS08B Ergonomic Viewing: Enjoy more viewing comfort while working at home or in the office. The laptop stand features a platform that is angled at 15° with 5.7” height in the back and 4.1” height in the front to bring your screen closer to eye level. One For All: Our laptop stand supports 10″ to 15.6″ laptops and tablets compatible with MacBook, HP, Lenovo, Dell, and more. Stable Support: The computer stand is designed with non-slip platform pads, padded front edges, and 4 non-slip base pads to keep your device securely in place. The Huanuo laptop riser can sturdily hold up to 8.8 lbs (4 kg). Faster Heat Dissipation: The laptop stand for desk features larger U-shaped ventilation for better airflow to keep your laptop cooler and avoid overheating to maximize device performance. Extra Space: The laptop stand features space underneath the platform to store 

In [22]:
response2 = cohere_client.rerank(
    model="rerank-v4.0-fast",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [23]:
reranked_results2 = [to_rerank[result.index] for result in response.results ]
reranked_results2

['HUANUO Laptop Stand, Ergonomic Laptop Stand for Desk, Notebook Computer Stand Holder Compatible with 10-15.6 Inch Laptops, Black, HNLS08B Ergonomic Viewing: Enjoy more viewing comfort while working at home or in the office. The laptop stand features a platform that is angled at 15° with 5.7” height in the back and 4.1” height in the front to bring your screen closer to eye level. One For All: Our laptop stand supports 10″ to 15.6″ laptops and tablets compatible with MacBook, HP, Lenovo, Dell, and more. Stable Support: The computer stand is designed with non-slip platform pads, padded front edges, and 4 non-slip base pads to keep your device securely in place. The Huanuo laptop riser can sturdily hold up to 8.8 lbs (4 kg). Faster Heat Dissipation: The laptop stand for desk features larger U-shaped ventilation for better airflow to keep your laptop cooler and avoid overheating to maximize device performance. Extra Space: The laptop stand features space underneath the platform to store 